[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaylink-coder/miss-yaya/blob/main/yaya-ai/notebooks/train_colab.ipynb)

# Yaya-125M — Reasoning & Compute Training

Trains Yaya-125M (129M params, GPT-2 scale) on the reasoning + math + calculator-tool dataset.

**Required:** Runtime → Change runtime type → **T4 GPU** (free tier works)

What Yaya learns here:
- `<|think|>step-by-step reasoning<|/think|>` — chain-of-thought
- `<|calc|>EXPRESSION<|/calc|>=RESULT` — calculator tool use
- Math: arithmetic, algebra, geometry, word problems
- Logic: syllogisms, deduction, sequences

Checkpoints saved to **Google Drive** — survive session resets.
Re-running this notebook auto-resumes from the latest checkpoint.

## 1. Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU: {name}  ({vram} GB VRAM)')
    print(f'PyTorch: {torch.__version__}')
    dtype_choice = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
    print(f'Best dtype: {dtype_choice}')
else:
    print('WARNING: No GPU found. Training will be extremely slow on CPU.')
    dtype_choice = 'float32'

## 2. Mount Google Drive (persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR  = '/content/drive/MyDrive/yaya-125m'
CKPT_DIR   = f'{DRIVE_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CKPT_DIR)

## 3. Clone repo + install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/jaylink-coder/miss-yaya.git'

if not os.path.exists('/content/miss-yaya'):
    !git clone {REPO_URL} /content/miss-yaya
else:
    !cd /content/miss-yaya && git pull

!pip install -q sentencepiece pyyaml

os.chdir('/content/miss-yaya/yaya-ai')
import sys
sys.path.insert(0, '/content/miss-yaya/yaya-ai')
print('Working dir:', os.getcwd())

## 4. Configure training
Points the config at Drive checkpoints and sets GPU-appropriate dtype.

In [ ]:
import yaml, glob

CFG_PATH = 'configs/training/sft_125m.yaml'
with open(CFG_PATH) as f:
    cfg = yaml.safe_load(f)

# Save to Drive
cfg['checkpointing']['save_dir'] = CKPT_DIR

# GPU-tuned batch sizes
# T4  (15 GB): batch=8, grad_accum=4 → effective=32
# A100 (40GB): batch=16, grad_accum=2 → effective=32
is_a100 = 'A100' in torch.cuda.get_device_name(0) if torch.cuda.is_available() else False
cfg['training']['per_device_batch_size']      = 16 if is_a100 else 8
cfg['training']['gradient_accumulation_steps'] = 2  if is_a100 else 4
cfg['training']['dtype'] = dtype_choice

# Use the reasoning dataset (already in repo)
cfg['data']['train_data'] = 'data/sft/yaya_reasoning_combined.jsonl'
cfg['data']['eval_data']  = 'data/sft/yaya_reasoning_combined.jsonl'

with open(CFG_PATH, 'w') as f:
    yaml.dump(cfg, f)

eff_batch = cfg['training']['per_device_batch_size'] * cfg['training']['gradient_accumulation_steps']
print(f"Config updated:")
print(f"  dtype:          {dtype_choice}")
print(f"  batch_size:     {cfg['training']['per_device_batch_size']}")
print(f"  grad_accum:     {cfg['training']['gradient_accumulation_steps']}")
print(f"  effective_batch:{eff_batch}")
print(f"  max_steps:      {cfg['training']['max_steps']}")
print(f"  save_dir:       {CKPT_DIR}")

## 5. Train Yaya-125M

In [ ]:
import glob

# Auto-resume from latest Drive checkpoint
ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
if ckpts:
    resume_flag = f'--resume {ckpts[-1]}'
    print(f'Resuming from: {ckpts[-1]}')
else:
    resume_flag = ''
    print('Starting from scratch (no checkpoint found)')

!WANDB_DISABLED=true WANDB_MODE=disabled python -u scripts/train_sft.py \
    --model_config configs/model/yaya_125m.yaml \
    --train_config configs/training/sft_125m.yaml \
    {resume_flag}

## 6. Test — Compute & Reasoning
After training, test that Yaya can compute and reason.

In [ ]:
import torch, glob, os, sys

from src.utils.config import load_model_config
from src.model.yaya_model import YayaForCausalLM
from src.tokenizer.tokenizer import YayaTokenizer, ASSISTANT_TOKEN
from src.inference.generator import TextGenerator, GenerationConfig
from src.inference.tool_generator import ToolAugmentedGenerator
from src.training.checkpointing import CheckpointManager

# Load latest checkpoint
ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
if not ckpts:
    print('No checkpoint found — run training first.')
else:
    ckpt_path = ckpts[-1]
    print(f'Loading: {ckpt_path}')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    cfg = load_model_config('configs/model/yaya_125m.yaml')
    model = YayaForCausalLM(cfg).to(device)

    ckpt_mgr = CheckpointManager(save_dir=os.path.dirname(ckpt_path))
    ckpt_mgr.load(model, checkpoint_path=ckpt_path)
    model.eval()

    tokenizer = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
    base_gen  = TextGenerator(model, tokenizer, device=device)
    tool_gen  = ToolAugmentedGenerator(base_gen, verbose=True)

    gen_cfg = GenerationConfig(
        max_new_tokens=300,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.3,
        do_sample=True,
    )

    SYSTEM = (
        'You are Yaya, a brilliant AI assistant. '
        'Think carefully and show your reasoning. '
        'Use <|calc|>EXPRESSION<|/calc|> for arithmetic.'
    )

    QUESTIONS = [
        'What is 47 × 83?',
        'Solve for x: 3x - 9 = 12.',
        'A car travels 90 km/h for 2.5 hours. How far does it go?',
        'If all mammals are warm-blooded and dolphins are mammals, are dolphins warm-blooded?',
        'What is 15% of 4,800?',
    ]

    for q in QUESTIONS:
        msgs = [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user',   'content': q},
        ]
        prompt = tokenizer.format_chat(msgs) + '\n' + ASSISTANT_TOKEN + '\n'
        response = tool_gen.generate(prompt, gen_cfg)

        print(f'Q: {q}')
        print(f'A: {response.strip()}')
        print('-' * 60)

## 7. Download checkpoint (optional)
Download the trained checkpoint to your local machine.

In [ ]:
import glob, shutil
from google.colab import files

ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
if ckpts:
    latest = ckpts[-1]
    zip_path = f'/content/yaya-125m-latest.zip'
    shutil.make_archive('/content/yaya-125m-latest', 'zip', latest)
    files.download(zip_path)
    print(f'Downloaded: {zip_path}')
else:
    print('No checkpoint to download.')